In [ ]:
import os
import random
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

class SRImplicitDataset(Dataset):
    def __init__(self, img_dir, max_images=100):
        self.files = sorted([
            os.path.join(img_dir, f)
            for f in os.listdir(img_dir)
            if f.endswith(".png") or f.endswith(".jpg")
        ])[:max_images]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        hr = TF.to_tensor(img)

        h, w = hr.shape[1:]

        # HARD CLAMP HR (NO SKIP)
        h = max(2, h)
        w = max(2, w)
        hr = TF.resize(hr, (h, w), antialias=True)

        scale = random.uniform(1.5, 4.0)

        # SAFE LR (MIN = 2)
        lr_h = max(2, int(h / scale))
        lr_w = max(2, int(w / scale))

        lr = TF.resize(hr, (lr_h, lr_w), antialias=True)

        return lr, hr

In [ ]:
#Load Dataset
import torch
import numpy as np
import cv2
import os
from ultralytics import YOLO
from PIL import Image
import torchvision.transforms.functional as TF

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load detector
detector = YOLO("C:/Users/Mardyson Justin/Thesis/yolov9c_visdrone_finetune15/weights/best.pt")

# Load SRNO
sr_model = SRNOInspired().to(device)
sr_model.load_state_dict(torch.load(
    "C:/Users/Mardyson Justin/Thesis/SRNO_Checkpoints5/80k_30e/best_model.pth",
    map_location=device
))
sr_model.eval()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SRNOInspired(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, hidden, 3, padding=1),
            nn.ReLU()
        )

        self.mlp = nn.Sequential(
            nn.Linear(hidden + 2, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 3)
        )

    def forward(self, lr, out_h, out_w):
        B, _, H_lr, W_lr = lr.shape

        # SAFE INPUT SIZE
        if H_lr < 2 or W_lr < 2:
            lr = F.interpolate(lr, size=(max(2, H_lr), max(2, W_lr)), mode="bilinear", align_corners=False)

        feat = self.encoder(lr)

        B, C, H, W = feat.shape

        # SAFE FEATURE SIZE
        if H < 2 or W < 2:
            feat = F.interpolate(feat, size=(max(2, H), max(2, W)), mode="bilinear", align_corners=False)
            B, C, H, W = feat.shape

        # SAFE OUTPUT SIZE
        out_h = max(2, out_h)
        out_w = max(2, out_w)

        # coordinate grid
        y = torch.linspace(-1, 1, out_h, device=lr.device)
        x = torch.linspace(-1, 1, out_w, device=lr.device)
        yy, xx = torch.meshgrid(y, x, indexing="ij")

        grid = torch.stack((xx, yy), dim=-1)
        grid = grid.unsqueeze(0).repeat(B, 1, 1, 1)

        # safe grid_sample
        feat_up = F.grid_sample(
            feat,
            grid,
            mode='bilinear',
            align_corners=True,
            padding_mode='border'
        )

        feat_flat = feat_up.permute(0, 2, 3, 1).reshape(-1, C)
        coords_flat = grid.reshape(-1, 2)

        inp = torch.cat([feat_flat, coords_flat], dim=1)
        out = self.mlp(inp)

        out = out.view(B, out_h, out_w, 3).permute(0, 3, 1, 2)

        return out

In [ ]:
def detect_sr_icro(
    image_path,
    tau_low=0.3,
    tau_high=0.6,
    alpha_max=4,
    min_alpha=1.5,
    k=5.0,
    epsilon=0.09,
    max_iter=4,
    save_output=False,
    output_path=None
):

    image = Image.open(image_path).convert("RGB")
    current_img = np.array(image)

    metrics_log = []

    for iteration in range(max_iter):

        results = detector(current_img)[0]
        boxes = results.boxes

        if boxes is None or len(boxes) == 0:
            print("No detections.")
            break

        confs_before = boxes.conf.cpu().numpy()
        avg_conf_before = np.mean(confs_before)

        updated_regions = 0

        for box in boxes:

            conf = float(box.conf)

            # skip high-confidence detections
            if conf >= tau_high:
                continue

            xyxy = box.xyxy.cpu().numpy()[0]
            x1, y1, x2, y2 = map(int, xyxy)

            # boundary safety
            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(current_img.shape[1], x2)
            y2 = min(current_img.shape[0], y2)

            roi = current_img[y1:y2, x1:x2]

            if roi.size == 0:
                continue

            # ----- Adaptive Scaling -----

            if conf <= tau_low:
                alpha = alpha_max
            else:
                alpha = 1 + k * (tau_high - conf)

            # clamp scaling range
            alpha = max(min_alpha, min(alpha, alpha_max))

            roi_pil = Image.fromarray(roi)
            lr_tensor = TF.to_tensor(roi_pil).unsqueeze(0).to(device)

            H, W = roi.shape[:2]

            if H > 128 or W > 128:
                continue

            target_h = int(H * alpha)
            target_w = int(W * alpha)

            with torch.no_grad():
                torch.cuda.empty_cache()
                sr = sr_model(lr_tensor, target_h, target_w)

            sr_img = TF.to_pil_image(
                sr.squeeze(0).clamp(0, 1).cpu()
            )

            sr_np = np.array(sr_img)

            # resize back to original ROI size
            sr_np_resized = cv2.resize(
                sr_np,
                (x2 - x1, y2 - y1),
                interpolation=cv2.INTER_CUBIC
            )

            current_img[y1:y2, x1:x2] = sr_np_resized
            updated_regions += 1

        # ----- Re-detection -----

        results_after = detector(current_img)[0]
        boxes_after = results_after.boxes

        if boxes_after is None or len(boxes_after) == 0:
            break

        confs_after = boxes_after.conf.cpu().numpy()
        avg_conf_after = np.mean(confs_after)

        delta_conf = avg_conf_after - avg_conf_before

        metrics_log.append({
            "iteration": iteration + 1,
            "regions_updated": updated_regions,
            "avg_conf_before": float(avg_conf_before),
            "avg_conf_after": float(avg_conf_after),
            "delta_conf": float(delta_conf)
        })

        print(f"\nIteration {iteration+1}")
        print(f"Updated Regions      : {updated_regions}")
        print(f"Avg Confidence Before: {avg_conf_before:.4f}")
        print(f"Avg Confidence After : {avg_conf_after:.4f}")
        print(f"Delta Confidence     : {delta_conf:.4f}")

        # ----- Convergence -----

        if abs(delta_conf) < epsilon:
            print("Converged.")
            break

        if avg_conf_after >= tau_high:
            print("Reached Stable High Confidence.")
            break

    if save_output and output_path is not None:
        cv2.imwrite(output_path, current_img)

    return current_img, metrics_log

In [ ]:
def detect_sr_icro_fixed(
    image_path,
    scale=2,                 # choose 2, 3, or 4
    tau_high=0.6,
    epsilon=0.09,
    max_iter=4,
    save_output=False,
    output_path=None
):

    assert scale in [2,3,4], "Scale must be 2, 3, or 4"

    image = Image.open(image_path).convert("RGB")
    current_img = np.array(image)

    metrics_log = []

    for iteration in range(max_iter):

        results = detector(current_img)[0]
        boxes = results.boxes

        if boxes is None or len(boxes) == 0:
            print("No detections.")
            break

        confs_before = boxes.conf.cpu().numpy()
        avg_conf_before = np.mean(confs_before)

        updated_regions = 0

        for box in boxes:

            conf = float(box.conf)

            # skip high confidence detections
            if conf >= tau_high:
                continue

            xyxy = box.xyxy.cpu().numpy()[0]
            x1, y1, x2, y2 = map(int, xyxy)

            roi = current_img[y1:y2, x1:x2]

            if roi.size == 0:
                continue

            roi_pil = Image.fromarray(roi)
            lr_tensor = TF.to_tensor(roi_pil).unsqueeze(0).to(device)

            H, W = roi.shape[:2]

            # fixed scale
            target_h = int(H * scale)
            target_w = int(W * scale)

            with torch.no_grad():
                sr = sr_model(lr_tensor, target_h, target_w)

            sr_img = TF.to_pil_image(
                sr.squeeze(0).clamp(0,1).cpu()
            )

            sr_np = np.array(sr_img)

            # resize back to original ROI size
            sr_np_resized = cv2.resize(
                sr_np,
                (x2 - x1, y2 - y1),
                interpolation=cv2.INTER_CUBIC
            )

            current_img[y1:y2, x1:x2] = sr_np_resized
            updated_regions += 1

        # ----- Re-detection -----

        results_after = detector(current_img)[0]
        boxes_after = results_after.boxes

        if boxes_after is None or len(boxes_after) == 0:
            break

        confs_after = boxes_after.conf.cpu().numpy()
        avg_conf_after = np.mean(confs_after)

        delta_conf = avg_conf_after - avg_conf_before

        metrics_log.append({
            "iteration": iteration + 1,
            "scale": scale,
            "regions_updated": updated_regions,
            "avg_conf_before": float(avg_conf_before),
            "avg_conf_after": float(avg_conf_after),
            "delta_conf": float(delta_conf)
        })

        print(f"\nIteration {iteration+1}")
        print(f"Scale Factor        : {scale}x")
        print(f"Updated Regions     : {updated_regions}")
        print(f"Avg Confidence Before: {avg_conf_before:.4f}")
        print(f"Avg Confidence After : {avg_conf_after:.4f}")
        print(f"Delta Confidence     : {delta_conf:.4f}")

        # ----- Convergence -----

        if abs(delta_conf) < epsilon:
            print("Converged.")
            break

        if avg_conf_after >= tau_high:
            print("Reached Stable High Confidence.")
            break

    if save_output and output_path is not None:
        cv2.imwrite(output_path, current_img)

    return current_img, metrics_log

In [ ]:
# just added parameters for passing in the qualitative test
def detect_sr_icro_from_array(
    current_img,
    tau_low=0.3,
    tau_high=0.6,
    alpha_max=4,
    min_alpha=1.5,
    k=5.0,
    epsilon=0.09,
    max_iter=4,
    save_output=False,
    output_path=None
):

     # ✅ FIX: ensure correct type
    if isinstance(current_img, list):
        current_img = current_img[0]
    if isinstance(current_img, torch.Tensor):
        current_img = current_img.detach().cpu().numpy()

    metrics_log = []

    for iteration in range(max_iter):

        results = detector(current_img)[0]
        boxes = results.boxes

        if boxes is None or len(boxes) == 0:
            print("No detections.")
            break

        confs_before = boxes.conf.cpu().numpy()
        avg_conf_before = np.mean(confs_before)

        updated_regions = 0

        for box in boxes:

            conf = float(box.conf)

            if conf >= tau_high:
                continue

            xyxy = box.xyxy.cpu().numpy()[0]
            x1, y1, x2, y2 = map(int, xyxy)

            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(current_img.shape[1], x2)
            y2 = min(current_img.shape[0], y2)

            roi = current_img[y1:y2, x1:x2]

            if roi.size == 0:
                continue

            if conf <= tau_low:
                alpha = alpha_max
            else:
                alpha = 1 + k * (tau_high - conf)

            alpha = max(min_alpha, min(alpha, alpha_max))

            roi_pil = Image.fromarray(roi)
            lr_tensor = TF.to_tensor(roi_pil).unsqueeze(0).to(device)

            H, W = roi.shape[:2]

            if H > 128 or W > 128:
                continue

            target_h = int(H * alpha)
            target_w = int(W * alpha)

            with torch.no_grad():
                torch.cuda.empty_cache()
                sr = sr_model(lr_tensor, target_h, target_w)

            sr_img = TF.to_pil_image(sr.squeeze(0).clamp(0, 1).cpu())
            sr_np = np.array(sr_img)

            sr_np_resized = cv2.resize(
                sr_np,
                (x2 - x1, y2 - y1),
                interpolation=cv2.INTER_CUBIC
            )

            current_img[y1:y2, x1:x2] = sr_np_resized
            updated_regions += 1

        results_after = detector(current_img)[0]
        boxes_after = results_after.boxes

        if boxes_after is None or len(boxes_after) == 0:
            break

        confs_after = boxes_after.conf.cpu().numpy()
        avg_conf_after = np.mean(confs_after)

        delta_conf = avg_conf_after - avg_conf_before

        metrics_log.append({
            "iteration": iteration + 1,
            "regions_updated": updated_regions,
            "avg_conf_before": float(avg_conf_before),
            "avg_conf_after": float(avg_conf_after),
            "delta_conf": float(delta_conf)
        })

        if abs(delta_conf) < epsilon:
            break

        if avg_conf_after >= tau_high:
            break

    if save_output and output_path is not None:
        cv2.imwrite(output_path, current_img)

    return current_img, metrics_log 

In [ ]:
def detect_sr_icro_fixed_from_array(
    current_img,
    scale=2,
    tau_high=0.6,
    epsilon=0.09,
    max_iter=4,
    save_output=False,
    output_path=None
):

    assert scale in [2,3,4], "Scale must be 2, 3, or 4"

    # ✅ FIX: ensure correct type
    if isinstance(current_img, list):
        current_img = current_img[0]
    if isinstance(current_img, torch.Tensor):
        current_img = current_img.detach().cpu().numpy()

    metrics_log = []

    for iteration in range(max_iter):

        results = detector(current_img)[0]
        boxes = results.boxes

        if boxes is None or len(boxes) == 0:
            print("No detections.")
            break

        confs_before = boxes.conf.cpu().numpy()
        avg_conf_before = np.mean(confs_before)

        updated_regions = 0

        for box in boxes:

            conf = float(box.conf)

            if conf >= tau_high:
                continue

            xyxy = box.xyxy.cpu().numpy()[0]
            x1, y1, x2, y2 = map(int, xyxy)

            roi = current_img[y1:y2, x1:x2]

            if roi.size == 0:
                continue

            roi_pil = Image.fromarray(roi)
            lr_tensor = TF.to_tensor(roi_pil).unsqueeze(0).to(device)

            H, W = roi.shape[:2]

            target_h = int(H * scale)
            target_w = int(W * scale)

            with torch.no_grad():
                sr = sr_model(lr_tensor, target_h, target_w)

            sr_img = TF.to_pil_image(sr.squeeze(0).clamp(0,1).cpu())
            sr_np = np.array(sr_img)

            sr_np_resized = cv2.resize(
                sr_np,
                (x2 - x1, y2 - y1),
                interpolation=cv2.INTER_CUBIC
            )

            current_img[y1:y2, x1:x2] = sr_np_resized
            updated_regions += 1

        results_after = detector(current_img)[0]
        boxes_after = results_after.boxes

        if boxes_after is None or len(boxes_after) == 0:
            break

        confs_after = boxes_after.conf.cpu().numpy()
        avg_conf_after = np.mean(confs_after)

        delta_conf = avg_conf_after - avg_conf_before

        metrics_log.append({
            "iteration": iteration + 1,
            "scale": scale,
            "regions_updated": updated_regions,
            "avg_conf_before": float(avg_conf_before),
            "avg_conf_after": float(avg_conf_after),
            "delta_conf": float(delta_conf)
        })

        if abs(delta_conf) < epsilon:
            break

        if avg_conf_after >= tau_high:
            break

    if save_output and output_path is not None:
        cv2.imwrite(output_path, current_img)

    return current_img, metrics_log

In [ ]:
# TEST CODE

import os
import time
import cv2
import torch
import pandas as pd
import numpy as np
from ultralytics import YOLO
from PIL import Image
import torchvision.transforms.functional as TF
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from thop import profile
import lpips

device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# PATHS
# -----------------------------
test_folder = "C:/Users/Mardyson Justin/Thesis/VisDrone2019-DET-test/images/"
excel_path = "C:/Users/Mardyson Justin/Thesis/UpdIFinale/180k/finalized_object_test_180_c025_k5_e09.xlsx"

# GT PATH
gt_folder = "C:/Users/Mardyson Justin/Thesis/VisDrone2019-DET-test/annotations/"

os.makedirs(os.path.dirname(excel_path), exist_ok=True)

# -----------------------------
# LOAD MODELS
# -----------------------------
detector = YOLO("C:/Users/Mardyson Justin/Thesis/yolov9c_visdrone_finetune15/weights/best.pt")

sr_model = SRNOInspired().to(device)
sr_model.load_state_dict(torch.load(
"C:/Users/Mardyson Justin/Thesis/SRNO_Checkpoints5/180k_30e/best_model.pth",
map_location=device
))
sr_model.eval()

lpips_model = lpips.LPIPS(net='alex').cpu()
lpips_model.eval()

# -----------------------------
# FLOPS
# -----------------------------
dummy_lr = torch.randn(1,3,64,64).to(device)

try:
    base_flops,_ = profile(sr_model, inputs=(dummy_lr,256,256), verbose=False)
    base_flops /= 1e9
except:
    base_flops = None

dummy_yolo = torch.randn(1,3,640,640).to(device)

try:
    detector.model.to(device)
    yolo_flops,_ = profile(detector.model, inputs=(dummy_yolo,), verbose=False)
    yolo_flops /= 1e9
except:
    yolo_flops = None

# -----------------------------
# IOU
# -----------------------------
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter = max(0,xB-xA)*max(0,yB-yA)

    areaA = (boxA[2]-boxA[0])*(boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0])*(boxB[3]-boxB[1])

    union = areaA + areaB - inter
    if union == 0:
        return 0

    return inter/union

# -----------------------------
# MATCHING CONF
# -----------------------------
def get_matching_conf(box, results):

    if results.boxes is None:
        return 0

    boxes = results.boxes.xyxy.cpu().numpy()
    confs = results.boxes.conf.cpu().numpy()

    best_iou = 0
    best_conf = 0
    best_dist = 1e9
    best_center_conf = 0

    x1,y1,x2,y2 = box
    cx = (x1+x2)/2
    cy = (y1+y2)/2

    for b,c in zip(boxes,confs):

        bx1,by1,bx2,by2 = b

        iou = compute_iou(box,b)
        if iou > best_iou:
            best_iou = iou
            best_conf = c

        bx = (bx1+bx2)/2
        by = (by1+by2)/2

        dist = np.sqrt((cx-bx)**2 + (cy-by)**2)

        if dist < best_dist:
            best_dist = dist
            best_center_conf = c

    if best_iou > 0.05:
        return float(best_conf)

    if best_dist < 50:
        return float(best_center_conf)

    return 0

# -----------------------------
# MATCHING CLASS
# -----------------------------
def get_matching_class(box, results):

    if results.boxes is None:
        return -1

    boxes = results.boxes.xyxy.cpu().numpy()
    classes = results.boxes.cls.cpu().numpy()

    best_iou = 0
    best_cls = -1
    best_dist = 1e9
    best_center_cls = -1

    x1,y1,x2,y2 = box
    cx = (x1+x2)/2
    cy = (y1+y2)/2

    for b,c in zip(boxes,classes):

        bx1,by1,bx2,by2 = b

        iou = compute_iou(box,b)
        if iou > best_iou:
            best_iou = iou
            best_cls = c

        bx = (bx1+bx2)/2
        by = (by1+by2)/2

        dist = np.sqrt((cx-bx)**2 + (cy-by)**2)

        if dist < best_dist:
            best_dist = dist
            best_center_cls = c

    if best_iou > 0.05:
        return int(best_cls)

    if best_dist < 50:
        return int(best_center_cls)

    return -1

# -----------------------------
# GT MATCHING
# -----------------------------
def get_gt_class(box, gt_boxes, gt_classes):

    best_iou = 0
    best_cls = -1

    for b, c in zip(gt_boxes, gt_classes):
        iou = compute_iou(box, b)

        if iou > best_iou:
            best_iou = iou
            best_cls = c

    if best_iou > 0.5:
        return int(best_cls)

    return -1

# -----------------------------
# SIZE CATEGORY
# -----------------------------
def size_category(w,h):
    area = w*h
    if area < 1024:
        return "tiny"
    elif area < 9216:
        return "small"
    else:
        return "medium"

# -----------------------------
# OBJECT METRICS
# -----------------------------
def compute_object_metrics(sr_img, hr_img, box):

    x1, y1, x2, y2 = map(int, box)
    H, W = hr_img.shape[:2]

    x1 = max(0, min(x1, W - 2))
    x2 = max(x1 + 1, min(x2, W))
    y1 = max(0, min(y1, H - 2))
    y2 = max(y1 + 1, min(y2, H))

    crop_sr = sr_img[y1:y2, x1:x2]
    crop_hr = hr_img[y1:y2, x1:x2]

    h, w = crop_hr.shape[:2]

    if h < 7 or w < 7:
        return None, None, None

    try:
        psnr = peak_signal_noise_ratio(crop_hr, crop_sr, data_range=255)
    except:
        psnr = None

    try:
        ssim = structural_similarity(
            crop_hr,
            crop_sr,
            channel_axis=2,
            data_range=255,
            win_size=7
        )
    except:
        ssim = None

    try:
        MIN_SIZE = 32

        if h < MIN_SIZE or w < MIN_SIZE:
            crop_sr_lp = cv2.resize(crop_sr, (MIN_SIZE, MIN_SIZE), interpolation=cv2.INTER_CUBIC)
            crop_hr_lp = cv2.resize(crop_hr, (MIN_SIZE, MIN_SIZE), interpolation=cv2.INTER_CUBIC)
        else:
            crop_sr_lp = crop_sr
            crop_hr_lp = crop_hr

        t1 = TF.to_tensor(crop_sr_lp).unsqueeze(0) * 2 - 1
        t2 = TF.to_tensor(crop_hr_lp).unsqueeze(0) * 2 - 1

        with torch.no_grad():
            lp = lpips_model(t1, t2).item()

    except:
        lp = None

    return psnr, ssim, lp

# -----------------------------
# COUNT SR PASSES
# -----------------------------
def count_sr_passes(metrics_log):
    total = 0
    for m in metrics_log:
        total += m["regions_updated"]
    return max(1, total)


all_images = sorted([
    f for f in os.listdir(test_folder)
    if f.lower().endswith((".jpg",".png",".jpeg"))
])

rows = []

for img_name in all_images:

    print("Processing:",img_name)

    img_path = os.path.join(test_folder,img_name)

    hr_img = Image.open(img_path).convert("RGB")
    hr_np = np.array(hr_img)

    H,W = hr_np.shape[:2]

    gt_path = os.path.join(gt_folder, img_name.replace(".jpg", ".txt"))

    gt_boxes = []
    gt_classes = []

    if os.path.exists(gt_path):
        with open(gt_path, "r") as f:
            for line in f.readlines():
                parts = line.strip().split(",")
                if len(parts) < 6: continue
                
                x, y, w, h = map(float, parts[:4])
                cls_gt_raw = int(parts[5])

                if 1 <= cls_gt_raw <= 10:
                    cls_gt = cls_gt_raw - 1 
                    gt_boxes.append([x, y, x + w, y + h])
                    gt_classes.append(cls_gt)

    lr = cv2.resize(hr_np, (int(max(2, W / 1.5)), int(max(2, H / 1.5))), interpolation=cv2.INTER_AREA)

    lr_results = detector(lr, imgsz=640, conf=0.00)[0]

    start=time.perf_counter()
    yolo_results = detector(hr_np,imgsz=640)[0]
    yolo_results_lr = detector(lr, imgsz=640)[0]
    torch.cuda.empty_cache()
    runtime_yolo_img = time.perf_counter()-start

    if yolo_results.boxes is None:
        continue

    base_boxes = yolo_results.boxes.xyxy.cpu().numpy()
    base_classes = yolo_results.boxes.cls.cpu().numpy()

    num_objects = len(base_boxes)

    lr_tensor = TF.to_tensor(lr).unsqueeze(0).to(device)

    start=time.perf_counter()
    with torch.no_grad():
        sr = sr_model(lr_tensor,H,W)
    torch.cuda.empty_cache()    
    runtime_sr_img = time.perf_counter()-start

    sr_np = np.array(TF.to_pil_image(sr.squeeze().clamp(0,1).cpu()))
    sr_results = detector(sr_np,imgsz=640)[0]

    start=time.perf_counter()
    sr_fixed, log_fixed = detect_sr_icro_fixed(img_path,save_output=False)
    torch.cuda.empty_cache()
    runtime_icro_fixed_img = time.perf_counter()-start
    sr_fixed_results = detector(sr_fixed,imgsz=640)[0]

    start=time.perf_counter()
    sr_adaptive, log_adapt = detect_sr_icro(img_path,save_output=False)
    torch.cuda.empty_cache()
    runtime_icro_adapt_img = time.perf_counter()-start
    sr_adaptive_results = detector(sr_adaptive,imgsz=640)[0]
    torch.cuda.empty_cache()

    N_sr = 1
    N_fixed = count_sr_passes(log_fixed)
    N_adapt = count_sr_passes(log_adapt)

    total_area = H * W

    for i,(box,cls) in enumerate(zip(base_boxes,base_classes)):

        x1,y1,x2,y2 = map(int,box)
        w=x2-x1
        h=y2-y1

        if w < 4 or h < 4:
            continue

        size=size_category(w,h)

        gt_class = get_gt_class(box, gt_boxes, gt_classes)

        roi_area = w * h
        area_ratio = roi_area / total_area

        runtime_yolo = runtime_yolo_img / num_objects

        runtime_sr = (runtime_yolo_img + (runtime_sr_img * area_ratio) + runtime_yolo_img) / num_objects
        runtime_icro_fixed = (runtime_yolo_img + (runtime_icro_fixed_img * area_ratio) + runtime_yolo_img) / num_objects
        runtime_icro_adaptive = (runtime_yolo_img + (runtime_icro_adapt_img * area_ratio) + runtime_yolo_img) / num_objects

        if yolo_flops is not None:
            flops_yolo = yolo_flops / num_objects
        else:
            flops_yolo = None

        if base_flops is not None and yolo_flops is not None:
            flops_sr = (yolo_flops + (base_flops * N_sr * area_ratio) + yolo_flops) / num_objects
            flops_fixed = (yolo_flops + (base_flops * N_fixed * area_ratio) + yolo_flops) / num_objects
            flops_adapt = (yolo_flops + (base_flops * N_adapt * area_ratio) + yolo_flops) / num_objects
        else:
            flops_sr = None
            flops_fixed = None
            flops_adapt = None

        conf_yolo = get_matching_conf(box, yolo_results_lr)

        conf_sr=get_matching_conf(box,sr_results)
        conf_fixed=get_matching_conf(box,sr_fixed_results)
        conf_adapt=get_matching_conf(box,sr_adaptive_results)

        psnr_sr,ssim_sr,lpips_sr = compute_object_metrics(sr_np,hr_np,box)
        psnr_fixed,ssim_fixed,lpips_fixed = compute_object_metrics(sr_fixed,hr_np,box)
        psnr_adapt,ssim_adapt,lpips_adapt = compute_object_metrics(sr_adaptive,hr_np,box)

        initial_det_sr = get_matching_class(box, lr_results)
        final_det_sr = get_matching_class(box, sr_results)

        initial_det_fixed = get_matching_class(box, lr_results)
        final_det_fixed = get_matching_class(box, sr_fixed_results)

        initial_det_adapt = get_matching_class(box, lr_results)
        final_det_adapt = get_matching_class(box, sr_adaptive_results)

        rows.append({
        "image_name":img_name,
        "object_id":i,
        "size":size,

        "gt_class": gt_class,
        "class": initial_det_sr,

        "initial_detection_sr": initial_det_sr,
        "final_detection_sr": final_det_sr,

        "initial_detection_icro_fixed": initial_det_fixed,
        "final_detection_icro_fixed": final_det_fixed,

        "initial_detection_icro_adaptive": initial_det_adapt,
        "final_detection_icro_adaptive": final_det_adapt,

        "conf_yolo":conf_yolo,
        "conf_sr":conf_sr,
        "conf_icro_fixed":conf_fixed,
        "conf_icro_adaptive":conf_adapt,

        "runtime_yolo":runtime_yolo,
        "runtime_sr":runtime_sr,
        "runtime_icro_fixed":runtime_icro_fixed,
        "runtime_icro_adaptive":runtime_icro_adaptive,

        "flops_yolo": flops_yolo,
        "flops_sr":flops_sr,
        "flops_icro_fixed":flops_fixed,
        "flops_icro_adaptive":flops_adapt,

        "passes_icro_fixed": N_fixed,
        "passes_icro_adaptive": N_adapt,

        "psnr_sr":psnr_sr,
        "ssim_sr":ssim_sr,
        "lpips_sr":lpips_sr,

        "psnr_icro_fixed":psnr_fixed,
        "ssim_icro_fixed":ssim_fixed,
        "lpips_icro_fixed":lpips_fixed,

        "psnr_icro_adaptive":psnr_adapt,
        "ssim_icro_adaptive":ssim_adapt,
        "lpips_icro_adaptive":lpips_adapt
        })

df=pd.DataFrame(rows)
df.to_excel(excel_path,index=False)

print("Validation complete")

In [ ]:
import os
import cv2
import torch
import numpy as np
import io
import pandas as pd
from ultralytics import YOLO
from PIL import Image
import torchvision.transforms.functional as TF
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import lpips
import ipywidgets as widgets
from IPython.display import display

# -----------------------------
# DEVICE
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# OUTPUT DIRS
# -----------------------------
base_dir = "SimulationOutputs/"
roi_dir = os.path.join(base_dir, "roi_comparisons")
os.makedirs(base_dir, exist_ok=True)
os.makedirs(roi_dir, exist_ok=True)

# -----------------------------
# LOAD MODELS
# -----------------------------
detector = YOLO("C:/Users/Mardyson Justin/Thesis/yolov9c_visdrone_finetune15/weights/best.pt").to(device)

sr_model = SRNOInspired().to(device)
sr_model.load_state_dict(torch.load(
    "C:/Users/Mardyson Justin/Thesis/SRNO_Checkpoints5/180k_30e/best_model.pth",
    map_location=device
))
sr_model.eval()

lpips_model = lpips.LPIPS(net='alex').to(device)
lpips_model.eval()

# -----------------------------
# FILE UPLOADER
# -----------------------------
uploader = widgets.FileUpload(accept='image/*', multiple=False)
display(uploader)

print("Upload an image above")

# -----------------------------
# SAFE UPLOADER
# -----------------------------
def get_uploaded_content(uploader):
    uploaded = uploader.value
    if isinstance(uploaded, dict):
        return list(uploaded.values())[0]['content']
    elif isinstance(uploaded, (list, tuple)):
        return uploaded[0]['content']
    else:
        raise TypeError("Unsupported uploader format")

# -----------------------------
# IOU + CONF
# -----------------------------
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter = max(0, xB-xA) * max(0, yB-yA)
    if inter == 0:
        return 0

    areaA = (boxA[2]-boxA[0])*(boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0])*(boxB[3]-boxB[1])

    return inter / float(areaA + areaB - inter)

def get_best_confidence(ref_box, results):
    if results.boxes is None:
        return 0.0

    best_iou = 0
    best_conf = 0.0

    for box in results.boxes:
        b = list(map(int, box.xyxy.cpu().numpy()[0]))
        iou = compute_iou(ref_box, b)

        if iou > best_iou:
            best_iou = iou
            best_conf = float(box.conf)

    return best_conf

# -----------------------------
# METRICS PER ROI
# -----------------------------
def compute_metrics(ref, test):
    if ref.size == 0 or test.size == 0:
        return 0, 0, 0

    ref_gray = cv2.cvtColor(ref, cv2.COLOR_RGB2GRAY)
    test_gray = cv2.cvtColor(test, cv2.COLOR_RGB2GRAY)

    psnr = peak_signal_noise_ratio(ref_gray, test_gray)
    ssim = structural_similarity(ref_gray, test_gray)

    t1 = TF.to_tensor(ref).unsqueeze(0).to(device)
    t2 = TF.to_tensor(test).unsqueeze(0).to(device)

    with torch.no_grad():
        lp = lpips_model(t1, t2).item()

    return psnr, ssim, lp

# -----------------------------
# PIPELINE
# -----------------------------
def run_pipeline(change):

    if len(uploader.value) == 0:
        return

    print("Processing image...")

    content = get_uploaded_content(uploader)

    hr = Image.open(io.BytesIO(content)).convert("RGB")
    hr_np = np.array(hr)

    h, w = hr_np.shape[:2]
    hr_np = cv2.resize(hr_np, (540, int(h*(540/w))))

    # YOLO
    with torch.no_grad():
        yolo_res = detector(hr_np, imgsz=640)[0]

    # SR
    lr_tensor = TF.to_tensor(hr_np).unsqueeze(0).to(device).float()
    with torch.no_grad():
        sr = sr_model(lr_tensor, hr_np.shape[0]*2, hr_np.shape[1]*2)

    sr_np = np.array(TF.to_pil_image(sr.squeeze().cpu()))
    sr_np = cv2.resize(sr_np, (hr_np.shape[1], hr_np.shape[0]))

    with torch.no_grad():
        sr_res = detector(sr_np, imgsz=640)[0]

    # ICRO
    sr_fixed = detect_sr_icro_fixed_from_array(hr_np)
    sr_adapt = detect_sr_icro_from_array(hr_np)

    with torch.no_grad():
        fixed_res = detector(sr_fixed, imgsz=640)[0]
        adapt_res = detector(sr_adapt, imgsz=640)[0]

    # -----------------------------
    # EXCEL DATA
    # -----------------------------
    rows = []

    if yolo_res.boxes is not None:

        for i, box in enumerate(yolo_res.boxes):

            x1, y1, x2, y2 = map(int, box.xyxy.cpu().numpy()[0])
            ref_box = [x1, y1, x2, y2]

            crop_yolo = hr_np[y1:y2, x1:x2]
            crop_sr = sr_np[y1:y2, x1:x2]
            crop_fixed = sr_fixed[y1:y2, x1:x2]
            crop_adapt = sr_adapt[y1:y2, x1:x2]

            if crop_yolo.size == 0:
                continue

            # CONF
            c_yolo = float(box.conf)
            c_sr = get_best_confidence(ref_box, sr_res)
            c_fx = get_best_confidence(ref_box, fixed_res)
            c_ad = get_best_confidence(ref_box, adapt_res)

            # METRICS
            psnr_sr, ssim_sr, lp_sr = compute_metrics(crop_yolo, crop_sr)
            psnr_fx, ssim_fx, lp_fx = compute_metrics(crop_yolo, crop_fixed)
            psnr_ad, ssim_ad, lp_ad = compute_metrics(crop_yolo, crop_adapt)

            rows.append({
                "object_id": i,
                "conf_yolo": c_yolo,
                "conf_sr": c_sr,
                "conf_fixed": c_fx,
                "conf_adaptive": c_ad,
                "gain_sr": c_sr - c_yolo,
                "gain_fixed": c_fx - c_yolo,
                "gain_adaptive": c_ad - c_yolo,
                "psnr_sr": psnr_sr,
                "ssim_sr": ssim_sr,
                "lpips_sr": lp_sr,
                "psnr_fixed": psnr_fx,
                "ssim_fixed": ssim_fx,
                "lpips_fixed": lp_fx,
                "psnr_adaptive": psnr_ad,
                "ssim_adaptive": ssim_ad,
                "lpips_adaptive": lp_ad,
            })

    df = pd.DataFrame(rows)
    df.to_excel(os.path.join(base_dir, "object_metrics.xlsx"), index=False)

    print("Done! Excel saved → SimulationOutputs/object_metrics.xlsx")

# -----------------------------
# TRIGGER
# -----------------------------
uploader.observe(run_pipeline, names='value')